# Experimente

API-Anbindung und Durchführung der Experimente. Voraussetzung: `llm_client.py` mit `ask_openai`, `ask_google` und `frage` liegt im selben ORdner; `.env` enthält `OPEN_API_KEY` und `GOOGLE_API_KEY`.

_Kostenregel_: Mechanik immer mit Mini-Prompts testen. Vollen Datensatz (~115.000 Tokens) nur im finalen, geplanten Lauf.


## 1. Setup


In [1]:
import os, json, datetime
from pathlib import Path
from dotenv import load_dotenv

# beim Entwickeln praktisch: llm_client.py wird bei Änderung neu geladen
%load_ext autoreload
%autoreload 2

load_dotenv() # Look for a .env file in the same directory as the Python script


True

In [2]:
# Prüfen, dass beide Keys geladen sind 

print(os.environ.get('OPENAI_API_KEY', 'NICHT GEFUNDEN')[:7])
print(os.environ.get('GOOGLE_API_KEY', 'NICHT GEFUNDEN')[:7])

sk-proj
AQ.Ab8R


## 2. Modelle und System-Prompts


In [14]:
MODEL_OPENAI = 'gpt-5.6-luna'
MODEL_GOOGLE = 'gemini-3.6-flash'

MODELS = [('openai', MODEL_OPENAI), ('google', MODEL_GOOGLE)]

# neutraler System-Prompt, englisch
SYSTEM_PROMPT      = "You are a data analyst answering questions about a dataset provided to you. Answer factually and precisely."
SYSTEM_PROMPT_EXP0 = "You are a data analyst answering questions about exploratory data analysis. Answer factually and precisely."
SYSTEM_PROMPT_EXP1 = "You are a data analyst answering questions about exploratory data analysis. Answer the user's questions about the provided dataset factually and precisely."


ITERATIONS = 3

## 3. Funktionen


In [4]:
from llm_client import ask_openai, ask_google, create_google_cache, delete_google_cache

"""
Konstanter leading Block mit dem Dataset. Bleibt identisch über alle Requests damit es gecached wird (OpenAI prefix cache, Gemini context cache)
"""
def build_dataset_block(data_text):
    return f"Here is a dataset in CSV format:\n\n{data_text}"

def build_question(question_text):
    return f"Question: {question_text}"

def ask(provider, question_text, system_prompt=None, temperature=1.0, max_tokens=1500, model=None,
        data_block=None, prompt_cache_key=None, google_cache=None):

    if provider == 'openai':
        if data_block is not None:
            prompt = f"{data_block}\n\n{build_question(question_text)}"
        else:
            prompt = question_text
        kw = {'model': model} if model else {}
        return ask_openai(prompt, 
                          system_prompt=system_prompt, 
                          temperature=temperature, 
                          max_completion_tokens=max_tokens,
                          prompt_cache_key=prompt_cache_key,
                          **kw)

    elif provider == 'google':
        if google_cache:
            prompt = build_question(question_text)
        elif data_block is not None:
            prompt = f"{data_block}\n\n{build_question(question_text)}"
        else:
            prompt = question_text
        kw = {'model':model} if model else {}
        return ask_google(prompt, 
                          system_prompt=system_prompt, 
                          temperature=temperature, 
                          max_tokens=max_tokens,
                          cached_content=google_cache,
                          **kw)
    else:
        raise ValueError(f"Unbekannter Anbieter: {provider}")

"""
Baut den User Prompt aus einer Frage. Bei with_dataset=True wird der Datensatz vorangestellt. Das ist wichtig für das Caching: gleichbleibender Teil zuerst, wechselnde Frage zuletzt.
"""
# def build_prompt(question_text, with_dataset=False, data_text=None):
#     if with_dataset:
#         if data_text is None:
#             raise ValueError("with_dataset=True, but no data_text provided")
#         return (
#             "Here is a dataset in CSV format:\n\n"
#             f"{data_text}\n\n"
#             f"Question: {question_text}"
#         )
#     else:
#         return question_text

"""
Hängt einen answer-entry als JSON-Zeile an die JSONL-Datei an. Schützt vor Datenverlust beim Absturz (sofortiges Schreiben)
"""
def save_answer(file, entry):
    file = Path(file)
    file.parent.mkdir(parents=True, exist_ok=True)
    with open(file, 'a', encoding='utf-8') as f:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

"""Outputfile lesen und Return von question_id, provider und iteration, die schon eine erfolgreiche Antwort haben"""

def already_done(output_file):

    output_file = Path(output_file)
    done = set()
    if not output_file.exists():
        return done
    with open(output_file, encoding='utf-8') as f:
        for line in f:
            try:
                e = json.loads(line)
            except json.JSONDecodeError:
                continue
            # nur die entries zählen, die wirklich eine Antwort haben
            if e.get('answer') is not None:
                done.add((e.get('question_id'), e.get('provider'), e.get('iteration')))

    return done


"""
Führt ein Experiment durch: jede Frage x jedes model x Wiederholungen.
Jede Antwort wird sofort gespeichert. Frischer Kontext pro Aufruf.

Caching: OpenAi uses automatic prefix caching; Google uses one explicit context cache per google model,
created before the loop and deleted afterwards
"""
def runner(experiment, questions, models, iterations, system_prompt, with_dataset=False, data_text=None, output_file=None,
           max_tokens = 1500, cache_ttl_seconds=3600):
    if output_file is None:
        output_file = Path('results') / f'{experiment}_answers.jsonl'

    done = already_done(output_file)

    data_block = build_dataset_block(data_text) if with_dataset else None
    # one stable key per experiment+dataset keeps OpenAI requests on the same cache
    prompt_cache_key = f'{experiment}-dataset' if with_dataset else None

    total = len(questions) * len(models) * iterations
    count = 0
    google_caches = {}

    try:
        for provider, model in models:
            gcache = None
            if with_dataset and provider == 'google':
                gcache = create_google_cache(data_block, system_prompt,
                                             model=model,
                                             ttl_seconds=cache_ttl_seconds)
                google_caches[model] = gcache
            for q in questions:
                for it in range(1, iterations + 1):
                    count += 1
                    if (q['id'], provider, it) in done:
                        print(f"[{count}/{total}] {q['id']} {provider} it{it}: skip (bereits vorhanden)")
                        continue
                    try:
                        r = ask(provider,
                                q['question'],
                                system_prompt=system_prompt, 
                                model=model,
                                max_tokens=max_tokens,
                                data_block=data_block,
                                prompt_cache_key=prompt_cache_key,
                                google_cache=gcache)

                        entry = {
                            'experiment': experiment,
                            'question_id': q['id'],
                            'category': q.get('category'),
                            'iteration': it,
                            'provider': provider,
                            'model': r['model_requested'],
                            'model_version': r['model_version'],
                            'prompt': '[dataset + question]' if with_dataset else q['question'],
                            'answer': r['answer'],
                            'input_tokens': r['input_tokens'],
                            'cached_tokens': r.get('cached_tokens', 0),
                            'output_tokens': r['output_tokens'],
                            'temperature': r['temperature'],
                            'finish_reason': r['finish_reason'],
                            'timestamp': r['timestamp'],
                        }
                        save_answer(output_file, entry)
                        status = f"ok (cached {r.get('cached_tokens', 0)}/{r['input_tokens']})"
                    except Exception as e:
                        save_answer(output_file, {
                            'experiment': experiment,
                            'question_id': q['id'],
                            'iteration': it,
                            'provider': provider,
                            'model': model,
                            'error': str(e),
                        })
                        status = f'ERROR: {e}'
                    print(f"[{count}/{total}] {q['id']} {provider} it{it}: {status}")
    finally:
        # remove explicit Google caches
        for name in google_caches.values():
            delete_google_cache(name)

    print(f"\nFinished. Saved in {output_file}")

"""
Lade den Fragenkatalog
"""    
def load_questions(experiment):
    path = Path('fragenkataloge') / f'{experiment}.json'
    with open(path, encoding='utf-8') as f:
        return json.load(f)

## 4. Verbindungstest


In [5]:
for provider, model in MODELS:
    print(f'=== {provider} ===')
    try:
        r = ask(provider, 'Antworte mit genau einem Wort: funktioniert.', model=model)
        print('answer:', r['answer'])
        print('version:', r['model_version'])
        print('tokens:', r['input_tokens'], '/',
            r['output_tokens'])
    except Exception as e:
        print('Fehler:', e)
    print()

=== openai ===
answer: funktioniert
version: gpt-5.6-luna
tokens: 16 / 5

=== google ===
answer: funktioniert
version: gemini-3.6-flash
tokens: 11 / 3



## 5. Tests (ohne API)

Reiner String-Bau und Speicher-Test ohne Kosten


## 5. Tests ohne API


In [6]:
# Prompt-Bau: E0 (ohne Datensatz) und E1 (mit Mini-Datensatz)
print("=== E0 (ohne Datensatz) ===")
q0 ="What is the IQR method used for?"
print(q0)
print()

mini = "Age,MonthlyIncome,Attrition\n35,5000,No\n42,3000,Yes\n28,4500,No"
q1 = "How many rows does the dataset contain?"

# E1 wie OpenAI es sieht: Datensatzblock + Frage in einer Nachricht
print("=== E1 OpenAI (Datensatz im Prompt) ===")
print(f"{build_dataset_block(mini)}\n\n{build_question(q1)}")
print()

# E1 wie Gemini es sieht: Datensatz liegt im Cache, gesendet wird nur die Frage
print("=== E1 Gemini (Datensatz im Cache) ===")
print(build_question(q1))

=== E0 (ohne Datensatz) ===
What is the IQR method used for?

=== E1 OpenAI (Datensatz im Prompt) ===
Here is a dataset in CSV format:

Age,MonthlyIncome,Attrition
35,5000,No
42,3000,Yes
28,4500,No

Question: How many rows does the dataset contain?

=== E1 Gemini (Datensatz im Cache) ===
Question: How many rows does the dataset contain?


## 6. Datensatz Laden

Rohdatensatz als CSV-Text für die datenbehafteten Experimente (E1-E3): Prompts mit diesem Text kosten ~115.000 Input-Token pro Aufruf.


In [7]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path('data') / 'ibm_original.csv')
data_as_text = df.to_csv(index=False)

print('Zeilen', len(df), '| Spalten:', df.shape[1])

Zeilen 1470 | Spalten: 35


## 7. Experiment 0 - kompletter Lauf

EDA-Konzeptwissen, **ohne** Datensatz (billig). 15 Fragen × 2 Modelle × 3 Wiederholungen = 90 Aufrufe.
Wiederaufnahme ist an: ein Neustart überspringt bereits Beantwortetes.


In [8]:
questions_e0 = load_questions('experiment0')

runner(
    experiment='E0',
    questions=questions_e0,
    models=MODELS,
    iterations=ITERATIONS, #3
    system_prompt=SYSTEM_PROMPT_EXP0,
    with_dataset=False,
    output_file=Path('results') / 'E0_answers.jsonl'
)

[1/90] E0_01 openai it1: skip (bereits vorhanden)
[2/90] E0_01 openai it2: skip (bereits vorhanden)
[3/90] E0_01 openai it3: skip (bereits vorhanden)
[4/90] E0_02 openai it1: skip (bereits vorhanden)
[5/90] E0_02 openai it2: skip (bereits vorhanden)
[6/90] E0_02 openai it3: skip (bereits vorhanden)
[7/90] E0_03 openai it1: skip (bereits vorhanden)
[8/90] E0_03 openai it2: skip (bereits vorhanden)
[9/90] E0_03 openai it3: skip (bereits vorhanden)
[10/90] E0_04 openai it1: skip (bereits vorhanden)
[11/90] E0_04 openai it2: skip (bereits vorhanden)
[12/90] E0_04 openai it3: skip (bereits vorhanden)
[13/90] E0_05 openai it1: skip (bereits vorhanden)
[14/90] E0_05 openai it2: skip (bereits vorhanden)
[15/90] E0_05 openai it3: skip (bereits vorhanden)
[16/90] E0_06 openai it1: skip (bereits vorhanden)
[17/90] E0_06 openai it2: skip (bereits vorhanden)
[18/90] E0_06 openai it3: skip (bereits vorhanden)
[19/90] E0_07 openai it1: skip (bereits vorhanden)
[20/90] E0_07 openai it2: skip (bereits 

In [12]:
SYSTEM_PROMPT = "You are a data analysis assistant. Answer strictly based on the provided dataset."

# Pfad zu deinem echten Rohdatensatz anpassen
data_text = Path("data/ibm_original.csv").read_text(encoding="utf-8")

questions = [
    {"id": "MINI_01", "category": "A", "question": "How many rows does the dataset contain?"},
    {"id": "MINI_02", "category": "A", "question": "What is the average MonthlyIncome?"},
    {"id": "MINI_03", "category": "A", "question": "What is the attrition rate (share of Attrition = Yes)?"},
]

models = [("openai", "gpt-5.6-luna"), ("google", "gemini-3.6-flash")]

runner("E1_MINI", questions, models, iterations=2,
       system_prompt=SYSTEM_PROMPT, with_dataset=True, data_text=data_text,
       max_tokens=4000, output_file="results/E1_MINI_answers.jsonl")

print("\n--- cached_tokens ---")
rows = [json.loads(l) for l in open("results/E1_MINI_answers.jsonl", encoding="utf-8")]
for r in rows:
    if r.get("answer") is not None:
        print(r["provider"], r["question_id"], "it"+str(r["iteration"]),
              "| input", r["input_tokens"], "| cached", r.get("cached_tokens", 0))

[1/12] MINI_01 openai it1: ok (cached 115642/115645)
[2/12] MINI_01 openai it2: ok (cached 115642/115645)
    retry in 1s (Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-5.6-luna in organization org-DzC)
[3/12] MINI_02 openai it1: ok (cached 115641/115644)
    retry in 1s (Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-5.6-luna in organization org-DzC)
[4/12] MINI_02 openai it2: ok (cached 115641/115644)
    retry in 1s (Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-5.6-luna in organization org-DzC)
[5/12] MINI_03 openai it1: ok (cached 115648/115651)
[6/12] MINI_03 openai it2: ok (cached 115648/115651)
    retry in 1s (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high deman)
    retry in 2s (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high deman)
[7/12] MINI_01 google it1: ok (cached 136781/136792)
[8/12] MINI_01 google it2: ok (cac

## Experiment 1


In [ ]:
questions_e1 = load_questions('experiment1')

runner(
    experiment='E1',
    questions=questions_e1,
    models=MODELS,
    iterations=5, #3
    system_prompt=SYSTEM_PROMPT_EXP1,
    with_dataset=False,
    output_file=Path('results') / 'E1_answers.jsonl'
)